# TF-IDF va BM25 Search Engine

Notebook nay xay dung hai baseline search engine cho tap bao tieng Viet:

- TF-IDF + cosine similarity.
- BM25 voi inverted index.

Ban nay chi can Python standard library de chay core logic. `pandas` la tuy chon, chi dung de hien thi ket qua dang bang neu moi truong da cai san.

## 1. Import va cau hinh

Mac dinh notebook chi doc mot phan dataset de tranh ton RAM. Tang `MAX_DOCS` khi can index nhieu hon.

In [1]:
from __future__ import annotations

from collections import Counter, defaultdict
from pathlib import Path
import heapq
import json
import math
import re
import unicodedata

try:
    import pandas as pd
except ModuleNotFoundError:
    pd = None

In [2]:
DATA_PATH_CANDIDATES = [
    Path("../data/news_dataset.json"),
]
DATA_PATH = next((path for path in DATA_PATH_CANDIDATES if path.exists()), DATA_PATH_CANDIDATES[0])

# File hien tai kha lon, nen de 20k cho demo nhanh.
MAX_DOCS = 20_000

# Lap lai title de tang trong so cho ket qua co query nam trong title.
TITLE_WEIGHT = 3

# Tham so TF-IDF.
TFIDF_MIN_DF = 1
TFIDF_MAX_DF_RATIO = 0.90

# Tham so BM25 pho bien.
BM25_K1 = 1.5
BM25_B = 0.75

DATA_PATH.resolve()

WindowsPath('D:/projects/vietnamese-news-search-engine/data/news_dataset.json')

## 2. Helper hien thi

Neu co `pandas`, cac ham search tra ve DataFrame. Neu khong, ket qua van la list dictionary va van doc duoc trong notebook.

In [3]:
def as_table(rows):
    if pd is not None:
        return pd.DataFrame(rows)
    return rows


def as_text(value) -> str:
    if value is None:
        return ""
    return str(value)

## 3. Load dataset dang JSON lon

`data/news_dataset.json` la JSON array lon. Ham ben duoi doc streaming tung object, nen co the lay mau bang `MAX_DOCS` ma khong can load ca file vao RAM.

In [4]:
def iter_json_array(file_obj, limit: int | None = None, chunk_size: int = 1_048_576):
    decoder = json.JSONDecoder()
    buffer = ""
    pos = 0
    started = False
    count = 0
    eof = False

    while True:
        if not eof:
            chunk = file_obj.read(chunk_size)
            if chunk == "":
                eof = True
            buffer += chunk

        while True:
            n = len(buffer)

            while pos < n and buffer[pos].isspace():
                pos += 1

            if not started:
                if pos >= n:
                    break
                if buffer[pos] != "[":
                    raise ValueError("Expected a JSON array.")
                started = True
                pos += 1
                continue

            while pos < n and (buffer[pos].isspace() or buffer[pos] == ","):
                pos += 1

            if pos >= n:
                break
            if buffer[pos] == "]":
                return

            try:
                obj, end = decoder.raw_decode(buffer, pos)
            except json.JSONDecodeError:
                if eof:
                    raise
                break

            yield obj
            count += 1
            if limit is not None and count >= limit:
                return
            pos = end

        if eof:
            return

        buffer = buffer[pos:]
        pos = 0


def iter_json_records(path: Path, limit: int | None = None):
    path = Path(path)
    with path.open("r", encoding="utf-8", errors="replace") as f:
        prefix = f.read(4096)
        f.seek(0)
        first = next((ch for ch in prefix if not ch.isspace()), "")

        if first == "[":
            yield from iter_json_array(f, limit=limit)
            return

        count = 0
        for line in f:
            line = line.strip().rstrip(",")
            if not line:
                continue
            yield json.loads(line)
            count += 1
            if limit is not None and count >= limit:
                return


def load_dataset(path: Path, max_docs: int | None = None) -> list[dict]:
    seen = set()
    records = []

    for record in iter_json_records(path, limit=max_docs):
        key = record.get("url") or record.get("id")
        if key is not None:
            if key in seen:
                continue
            seen.add(key)

        records.append(record)

    return records


raw_documents = load_dataset(DATA_PATH, MAX_DOCS)
print(f"Loaded {len(raw_documents):,} raw documents")
as_table(raw_documents[:3])

Loaded 20,000 raw documents


,id,author,content,picture_count,processed,source,title,topic,url,crawled_at,title_processed,content_processed,combined_processed,combined_unaccented
0,218270,,"Chiều 31/7, Công an tỉnh Thừa Thiên - Huế đã c...",3,0,docbao.vn,"Tên cướp tiệm vàng tại Huế là đại uý công an, ...",Pháp luật,https://docbao.vn/phap-luat/ten-cuop-tiem-vang...,1659344962817,cướp tiệm vàng huế đại_uý công_an công_tác trạ...,chiều date0731 công_an tỉnh thừa_thiên - huế t...,cướp tiệm vàng huế đại_uý công_an công_tác trạ...,cuop tiem vang hue dai_uy cong_an cong_tac tra...
1,218269,(Nguồn: Sina),"Gần đây, Thứ trưởng Bộ Phát triển Kỹ thuật số,...",1,0,vtc.vn,"Bỏ qua mạng 5G, Nga tiến thẳng từ 4G lên 6G",Sống kết nối,https://vtc.vn/bo-qua-mang-5g-nga-tien-thang-t...,1659344961181,bỏ_qua mạng 5g nga tiến thẳng 4g 6g,thứ_trưởng phát_triển kỹ_thuật_số truyền_thông...,bỏ_qua mạng 5g nga tiến thẳng 4g 6g thứ_trưởng...,bo_qua mang 5g nga tien thang 4g 6g thu_truong...
2,218268,Hồ Sỹ Anh,Kết quả thi tốt nghiệp THPT năm 2022 cho thấy ...,3,0,thanhnien.vn,Địa phương nào đứng đầu cả nước tổng điểm 3 mô...,Giáo dục,https://thanhnien.vn/dia-phuong-nao-dung-dau-c...,1659344955311,địa_phương đứng đầu tổng 3 môn văn toán ngoại_ngữ,kết_quả thi tốt_nghiệp thpt 2022 trung_bình mô...,địa_phương đứng đầu tổng 3 môn văn toán ngoại_...,dia_phuong dung dau tong 3 mon van toan ngoai_...


## 4. Chuan hoa text cho index

Pipeline nay:

- Thu sua mojibake co dang `TÃªn`, `cÆ°á»›p` neu gap.
- Chuyen ve lowercase va bo dau de query co dau/khong dau deu match.
- Mo rong token co dau gach duoi: `bong_da` thanh ca `bong_da`, `bong`, `da`.
- Bo mot so boilerplate quang cao/thong tin trang.

In [5]:
MOJIBAKE_MARKERS = ("Ã", "Ä", "Æ", "Â", "â€", "áº", "á»")
TOKEN_RE = re.compile(r"[0-9a-zA-Z_]+")

BOILERPLATE_PATTERNS = [
    r"\(adsbygoogle.*?push\(\{\}\);?",
    r"adsbygoogle\s+window\s+adsbygoogle\s+push",
    r"https?://\S+",
    r"trang thong tin dien tu.*",
    r"chinh sach bao mat.*",
    r"rss$",
]


def mojibake_score(text: str) -> int:
    return sum(text.count(marker) for marker in MOJIBAKE_MARKERS)


def fix_mojibake(text: str) -> str:
    text = as_text(text)
    score = mojibake_score(text)
    if score == 0:
        return text

    candidates = [text]
    for encoding in ("latin1", "cp1252"):
        try:
            candidates.append(text.encode(encoding, errors="replace").decode("utf-8", errors="replace"))
        except UnicodeError:
            pass

    best = min(candidates, key=mojibake_score)
    return best if mojibake_score(best) < score else text


def strip_accents(text: str) -> str:
    text = text.replace("đ", "d").replace("Đ", "D")
    normalized = unicodedata.normalize("NFD", text)
    return "".join(ch for ch in normalized if unicodedata.category(ch) != "Mn")


def normalize_for_index(text: str) -> str:
    text = fix_mojibake(text).lower()
    text = strip_accents(text)

    for pattern in BOILERPLATE_PATTERNS:
        text = re.sub(pattern, " ", text, flags=re.IGNORECASE | re.DOTALL)

    tokens = TOKEN_RE.findall(text)
    expanded_tokens = []
    for token in tokens:
        if len(token) == 1 and not token.isdigit():
            continue

        expanded_tokens.append(token)
        if "_" in token:
            expanded_tokens.extend(part for part in token.split("_") if len(part) > 1)

    return " ".join(expanded_tokens)


def build_document_text(record: dict) -> str:
    title_parts = [record.get("title", ""), record.get("title_processed", "")]
    body_parts = [
        record.get("content_processed", ""),
        record.get("combined_processed", ""),
        record.get("combined_unaccented", ""),
        record.get("content", ""),
    ]

    weighted_parts = title_parts * TITLE_WEIGHT + body_parts
    return normalize_for_index(" ".join(as_text(part) for part in weighted_parts))


documents = []
for record in raw_documents:
    search_text = build_document_text(record)
    if not search_text:
        continue

    doc = dict(record)
    doc["search_text"] = search_text
    doc["tokens"] = search_text.split()
    documents.append(doc)

print(f"Indexable documents: {len(documents):,}")
as_table([{key: doc.get(key, "") for key in ["id", "title", "source", "topic"]} for doc in documents[:5]])

Indexable documents: 20,000


,id,title,source,topic
0,218270,"Tên cướp tiệm vàng tại Huế là đại uý công an, ...",docbao.vn,Pháp luật
1,218269,"Bỏ qua mạng 5G, Nga tiến thẳng từ 4G lên 6G",vtc.vn,Sống kết nối
2,218268,Địa phương nào đứng đầu cả nước tổng điểm 3 mô...,thanhnien.vn,Giáo dục
3,218267,Người chết trong mưa lũ 'nghìn năm có một' ở M...,vnexpress,Thế giới
4,218266,"Hải Phòng: Hình ảnh xe ""điên"" gây tai nạn liên...",soha,Thời sự - Xã hội


## 5. TF-IDF search engine

TF-IDF index ben duoi dung cong thuc:

- `tf = 1 + log(term_frequency)`.
- `idf = log((N + 1) / (df + 1)) + 1`.
- Vector document va query duoc L2-normalize, score la cosine similarity.

In [6]:
class TfidfIndex:
    def __init__(self, tokenized_docs, min_df: int = 1, max_df_ratio: float = 0.90):
        self.n_docs = len(tokenized_docs)
        self.inverted = defaultdict(list)
        self.idf = {}

        doc_freq = Counter()
        term_counts_by_doc = []

        for tokens in tokenized_docs:
            counts = Counter(tokens)
            term_counts_by_doc.append(counts)
            doc_freq.update(counts.keys())

        max_df = max(1, int(self.n_docs * max_df_ratio))
        vocabulary = {
            term
            for term, df in doc_freq.items()
            if df >= min_df and df <= max_df
        }

        self.idf = {
            term: math.log((self.n_docs + 1.0) / (doc_freq[term] + 1.0)) + 1.0
            for term in vocabulary
        }

        for doc_id, counts in enumerate(term_counts_by_doc):
            weights = {}
            for term, tf in counts.items():
                idf = self.idf.get(term)
                if idf is None:
                    continue
                weights[term] = (1.0 + math.log(tf)) * idf

            norm = math.sqrt(sum(weight * weight for weight in weights.values()))
            if norm == 0:
                continue

            for term, weight in weights.items():
                self.inverted[term].append((doc_id, weight / norm))

    def vectorize_query(self, query_tokens):
        counts = Counter(query_tokens)
        weights = {}
        for term, tf in counts.items():
            idf = self.idf.get(term)
            if idf is None:
                continue
            weights[term] = (1.0 + math.log(tf)) * idf

        norm = math.sqrt(sum(weight * weight for weight in weights.values()))
        if norm == 0:
            return {}

        return {term: weight / norm for term, weight in weights.items()}

    def search(self, query_tokens, top_k: int = 10):
        query_weights = self.vectorize_query(query_tokens)
        scores = defaultdict(float)

        for term, query_weight in query_weights.items():
            for doc_id, doc_weight in self.inverted.get(term, []):
                scores[doc_id] += query_weight * doc_weight

        return heapq.nlargest(top_k, scores.items(), key=lambda item: item[1])


tfidf_index = TfidfIndex(
    [doc["tokens"] for doc in documents],
    min_df=TFIDF_MIN_DF,
    max_df_ratio=TFIDF_MAX_DF_RATIO,
)
print(f"TF-IDF index: {tfidf_index.n_docs:,} docs, {len(tfidf_index.idf):,} terms")

TF-IDF index: 20,000 docs, 128,875 terms


In [15]:
a = [doc["tokens"] for doc in documents] 
len(a[0])

2310

## 6. BM25 search engine

BM25 duoc cai bang inverted index: moi term tro toi danh sach `(doc_id, term_frequency)`. Cach nay nhanh hon viec quet toan bo documents moi lan query.

In [7]:
class BM25Index:
    def __init__(self, tokenized_docs, k1: float = 1.5, b: float = 0.75):
        self.k1 = k1
        self.b = b
        self.n_docs = len(tokenized_docs)
        self.doc_len = [len(tokens) for tokens in tokenized_docs]
        self.avgdl = sum(self.doc_len) / self.n_docs if self.n_docs else 0.0

        self.inverted = defaultdict(list)
        doc_freq = Counter()

        for doc_id, tokens in enumerate(tokenized_docs):
            counts = Counter(tokens)
            for term, tf in counts.items():
                self.inverted[term].append((doc_id, tf))
                doc_freq[term] += 1

        self.idf = {
            term: math.log(1.0 + (self.n_docs - df + 0.5) / (df + 0.5))
            for term, df in doc_freq.items()
        }

    def search(self, query_tokens, top_k: int = 10):
        scores = defaultdict(float)
        if not query_tokens or self.avgdl == 0:
            return []

        for term in set(query_tokens):
            postings = self.inverted.get(term)
            if not postings:
                continue

            idf = self.idf.get(term, 0.0)
            for doc_id, tf in postings:
                dl = self.doc_len[doc_id]
                denom = tf + self.k1 * (1.0 - self.b + self.b * dl / self.avgdl)
                scores[doc_id] += idf * (tf * (self.k1 + 1.0)) / denom

        return heapq.nlargest(top_k, scores.items(), key=lambda item: item[1])


bm25_index = BM25Index([doc["tokens"] for doc in documents], k1=BM25_K1, b=BM25_B)
print(f"BM25 index: {bm25_index.n_docs:,} docs, {len(bm25_index.inverted):,} terms")

BM25 index: 20,000 docs, 128,878 terms


## 7. Ham search va hien thi ket qua

In [8]:
def prepare_query(query: str) -> list[str]:
    return normalize_for_index(query).split()


def display_text(value, max_chars: int | None = None) -> str:
    text = fix_mojibake(as_text(value))
    text = re.sub(r"\s+", " ", text).strip()
    if max_chars is not None and len(text) > max_chars:
        return text[:max_chars].rstrip() + "..."
    return text


def format_results(ranked, engine: str) -> list[dict]:
    rows = []
    for rank, (doc_id, score) in enumerate(ranked, start=1):
        doc = documents[int(doc_id)]
        rows.append(
            {
                "rank": rank,
                "engine": engine,
                "score": round(float(score), 6),
                "id": doc.get("id", ""),
                "title": display_text(doc.get("title", ""), 140),
                "source": doc.get("source", ""),
                "topic": doc.get("topic", ""),
                "snippet": display_text(doc.get("content", ""), 260),
                "url": doc.get("url", ""),
            }
        )
    return rows


def search_tfidf(query: str, top_k: int = 10):
    query_tokens = prepare_query(query)
    ranked = tfidf_index.search(query_tokens, top_k=top_k)
    return as_table(format_results(ranked, "tfidf"))


def search_bm25(query: str, top_k: int = 10):
    query_tokens = prepare_query(query)
    ranked = bm25_index.search(query_tokens, top_k=top_k)
    return as_table(format_results(ranked, "bm25"))


def compare_engines(query: str, top_k: int = 10):
    query_tokens = prepare_query(query)
    rows = []
    rows.extend(format_results(tfidf_index.search(query_tokens, top_k=top_k), "tfidf"))
    rows.extend(format_results(bm25_index.search(query_tokens, top_k=top_k), "bm25"))
    return as_table(rows)

## 8. Thu nghiem truy van

In [9]:
QUERY = "Liverpool Strasbourg du doan bong da"
search_tfidf(QUERY, top_k=5)

,rank,engine,score,id,title,source,topic,snippet,url
0,1,tfidf,0.354593,210930,"Soi kèo Liverpool vs Strasbourg, 1h30 ngày 1/8...",bongdaplus,Soi kèo,7/8 trận chính thức gần đây của Liverpool về x...,https://bongdaplus.vn/soi-keo/soi-keo-liverpoo...
1,2,tfidf,0.272787,218262,Soi kèo nhà cái Liverpool vs Strasbourg. Nhận ...,thethaovanhoa,,Soi kèo nhà cái Liverpool vs Strasbourg. Nhận ...,https://thethaovanhoa.vn/du-doan-bong-da/soi-k...
2,3,tfidf,0.272787,216498,Soi kèo nhà cái Liverpool vs Strasbourg. Nhận ...,thethaovanhoa,,Soi kèo nhà cái Liverpool vs Strasbourg. Nhận ...,https://thethaovanhoa.vn/du-doan-bong-da/soi-k...
3,4,tfidf,0.270725,209801,"Tỷ lệ kèo, keonhacai, soi kèo nhà cái, nhận đị...",thethaovanhoa,,"Tỷ lệ kèo, keonhacai, soi kèo nhà cái, nhận đị...",https://thethaovanhoa.vn/du-doan-bong-da/ty-le...
4,5,tfidf,0.243993,218117,Video bóng đá Liverpool - Strasbourg: Thảm họa...,24h.com.vn,Bóng đá,Do vừa thi đấu Siêu cúp Anh cùng Man City nên ...,https://www.24h.com.vn/bong-da/video-bong-da-l...


In [10]:
search_bm25(QUERY, top_k=5)

,rank,engine,score,id,title,source,topic,snippet,url
0,1,bm25,34.638835,218262,Soi kèo nhà cái Liverpool vs Strasbourg. Nhận ...,thethaovanhoa,,Soi kèo nhà cái Liverpool vs Strasbourg. Nhận ...,https://thethaovanhoa.vn/du-doan-bong-da/soi-k...
1,2,bm25,34.638835,216498,Soi kèo nhà cái Liverpool vs Strasbourg. Nhận ...,thethaovanhoa,,Soi kèo nhà cái Liverpool vs Strasbourg. Nhận ...,https://thethaovanhoa.vn/du-doan-bong-da/soi-k...
2,3,bm25,32.634024,208284,"Nhận định bóng đá Liverpool vs Strasbourg, 1h3...",bongdaplus,Giao hữu Quốc tế,"Do v?a ph?i ?� Community Shield, Liverpool s? ...",https://bongdaplus.vn/giao-huu-bong-da/nhan-di...
3,4,bm25,32.473242,209801,"Tỷ lệ kèo, keonhacai, soi kèo nhà cái, nhận đị...",thethaovanhoa,,"Tỷ lệ kèo, keonhacai, soi kèo nhà cái, nhận đị...",https://thethaovanhoa.vn/du-doan-bong-da/ty-le...
4,5,bm25,32.151665,218117,Video bóng đá Liverpool - Strasbourg: Thảm họa...,24h.com.vn,Bóng đá,Do vừa thi đấu Siêu cúp Anh cùng Man City nên ...,https://www.24h.com.vn/bong-da/video-bong-da-l...


In [11]:
compare_engines("cuop tiem vang hue cong an", top_k=5)

,rank,engine,score,id,title,source,topic,snippet,url
0,1,tfidf,0.413907,215953,"Cướp tiệm vàng ở Huế, không sợ súng, người dân...",danviet,,"Trưa 31/7, lực lượng Công an TP.Huế đã phong t...",https://danviet.vn/cuop-tiem-vang-o-hue-khong-...
1,2,tfidf,0.390512,217762,Công an đã bắt kẻ dùng súng cướp tiệm vàng ở Huế,danviet,,"Chiều 31/7, lãnh đạo Công an tỉnh Thừa Thiên H...",https://danviet.vn/cong-an-da-bat-ke-dung-sung...
2,3,tfidf,0.383759,215940,Huế: Tên cướp táo tợn nổ súng cướp tiệm vàng ở...,laodong,,"Trưa 31.7, lực lượng Công an TP.Huế phong tỏa ...",https://laodong.vn/phap-luat/hue-ten-cuop-tao-...
3,4,tfidf,0.375098,217331,Bắt giữ nghi phạm xả súng cướp tiệm vàng ở Huế,dantri,Pháp luật,Một vụ xả súng cướp tiệm vàng vừa xảy ra trưa ...,https://dantri.com.vn/phap-luat/bat-giu-nghi-p...
4,5,tfidf,0.363214,215626,Vây bắt kẻ dùng súng cướp tiệm vàng tại chợ Đô...,vtc.vn,Tin nhanh 24h,"Chiều 31/7, lực lượng Công an TP Huế phong tỏa...",https://vtc.vn/thua-thien-hue-vay-bat-ke-dung-...
5,1,bm25,29.855681,217762,Công an đã bắt kẻ dùng súng cướp tiệm vàng ở Huế,danviet,,"Chiều 31/7, lãnh đạo Công an tỉnh Thừa Thiên H...",https://danviet.vn/cong-an-da-bat-ke-dung-sung...
6,2,bm25,29.850932,216605,Nhân chứng kể về vụ nổ súng cướp tiệm vàng ở Huế,zingnews,,"Trưa 31/7, Công an tỉnh Thừa Thiên - Huế đã bắ...",https://zingnews.vn/nhan-chung-ke-ve-vu-no-sun...
7,3,bm25,29.832665,216456,Phó Giám đốc Công an tỉnh trực tiếp thuyết phụ...,24h.com.vn,,Đối tượng dùng súng cướp tại tiệm vàng Hoàng Đ...,https://www.24h.com.vn/an-ninh-hinh-su/pho-gia...
8,4,bm25,29.773828,215798,Kẻ mang AK cướp tiệm vàng ở Huế đòi gặp phó gi...,zingnews,Pháp luật,"Trưa 31/7, Công an tỉnh Thừa Thiên - Huế phong...",https://zingnews.vn/ke-mang-ak-cuop-tiem-vang-...
9,5,bm25,29.718004,217737,Nhân chứng kể lại giây phút đối tượng nổ súng ...,soha,Pháp luật,"Qua trao đổi, bà H (người có cửa hàng cạnh tiệ...",https://soha.vn/nhan-chung-ke-lai-giay-phut-do...


## 9. Luu index tuy chon

Thu muc `data/indexes/` dang duoc ignore trong `.gitignore`, phu hop de luu artifact local. Chi bat `SAVE_INDEX = True` khi can tai lai index ma khong build lai.

In [ ]:
SAVE_INDEX = False

if SAVE_INDEX:
    import pickle

    index_dir = DATA_PATH.parent / "indexes"
    index_dir.mkdir(parents=True, exist_ok=True)

    document_metadata = []
    for doc in documents:
        document_metadata.append(
            {key: doc.get(key, "") for key in ["id", "title", "content", "source", "topic", "url", "crawled_at"]}
        )

    payload = {
        "documents": document_metadata,
        "tfidf_index": tfidf_index,
        "bm25_index": bm25_index,
        "config": {
            "data_path": str(DATA_PATH),
            "max_docs": MAX_DOCS,
            "title_weight": TITLE_WEIGHT,
            "bm25_k1": BM25_K1,
            "bm25_b": BM25_B,
        },
    }

    output_path = index_dir / "tfidf_bm25_search_index.pkl"
    with output_path.open("wb") as f:
        pickle.dump(payload, f)

    print(f"Saved index to {output_path.resolve()}")